In [23]:
import pandas as pd
path = "https://raw.githubusercontent.com/ulka29/paper/7aad80ab3fe4ad5ffeabffb7c30bcb70edb218bd/01_data/processed/df_cleaned.csv"# import Check dataset
df= pd.read_csv(path)
df.head()

,tweet_text,cyberbullying_type,clean_tweet,lemmatized_tweet,sentiment_scores,negative,positive,neutral,compound
0,"in other words #katandandre, your food was cra...",not_cyberbullying,words katandandre food crapilicious mkr,word katandandre food crapilicious mkr,"{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...",0.000,0.000,1.000,0.0000
1,why is #aussietv so white? #mkr #theblock #ima...,not_cyberbullying,aussietv white mkr theblock imacelebrityau tod...,aussietv white mkr theblock imacelebrityau tod...,"{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...",0.000,0.000,1.000,0.0000
2,@xochitlsuckkks a classy whore? or more red ve...,not_cyberbullying,classy whore red velvet cupcakes,classy whore red velvet cupcake,"{'neg': 0.32, 'neu': 0.481, 'pos': 0.199, 'com...",0.320,0.199,0.481,-0.4137
3,"@jason_gio meh. :p thanks for the heads up, b...",not_cyberbullying,meh p thanks heads concerned another angry dud...,meh p thanks head concerned another angry dude...,"{'neg': 0.243, 'neu': 0.607, 'pos': 0.15, 'com...",0.243,0.150,0.607,-0.4854
4,@rudhoeenglish this is an isis account pretend...,not_cyberbullying,isis account pretending kurdish account like i...,isi account pretending kurdish account like is...,"{'neg': 0.129, 'neu': 0.691, 'pos': 0.18, 'com...",0.129,0.180,0.691,0.0258


In [24]:
# ============================================================
# CELL 1: DATA PREPARATION
# TF-IDF + SENTIMENT ANALYSIS + SMOTE
# ============================================================

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from imblearn.over_sampling import SMOTE
from scipy.sparse import hstack
from collections import Counter
import matplotlib.pyplot as plt
import numpy as np

# ------------------------------------------------------------
# 1. TEXT, SENTIMENT FEATURES AND TARGET
# ------------------------------------------------------------

X_text = df['lemmatized_tweet'].fillna('')
# Sentiment Analysis features
sentiment_features = [
    'negative',
    'positive',
    'neutral',
    'compound'
]

X_sentiment = df[sentiment_features].fillna(0)

# Target variable
y = df['cyberbullying_type']

# ------------------------------------------------------------
# 2. LABEL ENCODING
# ------------------------------------------------------------

label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)

print("Classes:")
print(label_encoder.classes_)

# Label mapping
label_mapping = dict(
    enumerate(label_encoder.classes_)
)

print("\nLabel Mapping:")

for number, category in label_mapping.items():
    print(f"{number} → {category}")

# ------------------------------------------------------------
# 3. TRAIN-TEST SPLIT
# ------------------------------------------------------------

X_text_train, X_text_test, X_sent_train, X_sent_test, y_train, y_test = train_test_split(
    X_text,
    X_sentiment,
    y_encoded,
    test_size=0.30,
    random_state=42,
    stratify=y_encoded
)

print("\nTrain-Test Split")
print("-------------------------")
print("Training samples:", len(X_text_train))
print("Testing samples :", len(X_text_test))

# ------------------------------------------------------------
# 4. TF-IDF VECTORIZATION
# ------------------------------------------------------------

tfidf = TfidfVectorizer(
    max_features=60000,
    ngram_range=(1, 2)
)

X_train_tfidf = tfidf.fit_transform(X_text_train)
X_test_tfidf = tfidf.transform(X_text_test)

print("\nTF-IDF Features")
print("-------------------------")
print("X_train TF-IDF:", X_train_tfidf.shape)
print("X_test TF-IDF :", X_test_tfidf.shape)

# ------------------------------------------------------------
# 5. COMBINE TF-IDF + SENTIMENT FEATURES
# ------------------------------------------------------------

X_train_combined = hstack([
    X_train_tfidf,
    X_sent_train.values
])

X_test_combined = hstack([
    X_test_tfidf,
    X_sent_test.values
])

print("\nCombined Features")
print("-------------------------")
print("X_train:", X_train_combined.shape)
print("X_test :", X_test_combined.shape)

Classes:
['age' 'ethnicity' 'gender' 'not_cyberbullying' 'other_cyberbullying'
 'religion']

Label Mapping:
0 → age
1 → ethnicity
2 → gender
3 → not_cyberbullying
4 → other_cyberbullying
5 → religion

Train-Test Split
-------------------------
Training samples: 29851
Testing samples : 12794

TF-IDF Features
-------------------------
X_train TF-IDF: (29851, 60000)
X_test TF-IDF : (12794, 60000)

Combined Features
-------------------------
X_train: (29851, 60004)
X_test : (12794, 60004)


In [25]:
# ============================================================
# CELL 2: FEATURE SELECTION SETUP
# ============================================================

from sklearn.feature_selection import SelectKBest, chi2
from scipy.sparse import csr_matrix, hstack
import numpy as np

# Number of TEXT features to retain
K = 10000

print("Original TF-IDF features:", X_train_tfidf.shape[1])
print("Selected TF-IDF features :", K)

Original TF-IDF features: 60000
Selected TF-IDF features : 10000


In [26]:
# ============================================================
# CELL 3: EXPERIMENT A — NO FEATURE SELECTION
# ============================================================

X_train_A = X_train_tfidf
X_test_A = X_test_tfidf

print("Experiment A — No Selection")
print("----------------------------")
print("Train shape:", X_train_A.shape)
print("Test shape :", X_test_A.shape)

Experiment A — No Selection
----------------------------
Train shape: (29851, 60000)
Test shape : (12794, 60000)


In [27]:
X_train_A_combined = hstack([
    X_train_A,
    csr_matrix(X_sent_train.values)
])

X_test_A_combined = hstack([
    X_test_A,
    csr_matrix(X_sent_test.values)
])

print("Final train shape:", X_train_A_combined.shape)
print("Final test shape :", X_test_A_combined.shape)

Final train shape: (29851, 60004)
Final test shape : (12794, 60004)


In [28]:
# ============================================================
# CELL 4: EXPERIMENT B — GLOBAL FEATURE SELECTION
# ============================================================

global_selector = SelectKBest(
    score_func=chi2,
    k=K
)

# IMPORTANT:
# Fit only on training data
X_train_B = global_selector.fit_transform(
    X_train_tfidf,
    y_train
)

# Apply the SAME selector to test data
X_test_B = global_selector.transform(
    X_test_tfidf
)

print("Experiment B — Global Selection")
print("--------------------------------")
print("Train shape:", X_train_B.shape)
print("Test shape :", X_test_B.shape)

Experiment B — Global Selection
--------------------------------
Train shape: (29851, 10000)
Test shape : (12794, 10000)


In [29]:
X_train_B_combined = hstack([
    X_train_B,
    csr_matrix(X_sent_train.values)
])

X_test_B_combined = hstack([
    X_test_B,
    csr_matrix(X_sent_test.values)
])

print("Final train shape:", X_train_B_combined.shape)
print("Final test shape :", X_test_B_combined.shape)

Final train shape: (29851, 10004)
Final test shape : (12794, 10004)


In [30]:
# ============================================================
# CELL 5: TOP GLOBAL FEATURES
# ============================================================

feature_names = np.array(tfidf.get_feature_names_out())

global_selected_features = feature_names[
    global_selector.get_support()
]

print("Number of selected features:",
      len(global_selected_features))

print("\nFirst 100 selected features:")
print(global_selected_features[:100])

Number of selected features: 10000

First 100 selected features:
['abdul' 'abdul kalam' 'abeed' 'able bully' 'able help' 'abortion'
 'abortion joke' 'abortion provider' 'abortion rape' 'abroad' 'absolute'
 'absolute idiot' 'absolutely' 'absolutely fucking' 'absorbed' 'absurd'
 'abt rape' 'abu' 'abuse' 'abuse bullying' 'abuse gay' 'abuse happens'
 'abuse joke' 'abuse policy' 'abuse social' 'abuse woman' 'abusive' 'acc'
 'accept' 'accept radical' 'acceptable' 'accepts' 'accomplished' 'account'
 'account bully' 'account sure' 'accts' 'accurate' 'accused rape' 'ache'
 'acho' 'acne' 'acquittal' 'acquittal christian' 'act' 'act dumb'
 'act like' 'act white' 'acted' 'acted like' 'acting' 'acting dumb'
 'acting like' 'action' 'action idiot' 'active' 'active member' 'activity'
 'actor' 'actual fuck' 'actually bullied' 'actually care' 'actually dumb'
 'actually islam' 'actually want' 'ad pro' 'adam baldwin' 'add facebook'
 'add muslim' 'ade' 'adjective' 'adjust' 'administration' 'admit quran'
 '

In [31]:
global_scores = global_selector.scores_

top_global_indices = np.argsort(global_scores)[::-1][:50]

print("\nTop 50 global features:")
for idx in top_global_indices:
    print(
        f"{feature_names[idx]:40s} "
        f"score = {global_scores[idx]:.4f}"
    )


Top 50 global features:
nigger                                   score = 1969.6107
school                                   score = 1938.7753
dumb                                     score = 1793.2850
fuck                                     score = 1623.0540
joke                                     score = 1552.7115
high                                     score = 1465.6002
high school                              score = 1443.3080
rape                                     score = 1265.1103
muslim                                   score = 1240.9083
bullied                                  score = 1179.9012
gay                                      score = 1140.8088
girl                                     score = 1047.5216
bully                                    score = 1007.6620
rape joke                                score = 846.8217
dumb as                                  score = 792.1110
as                                       score = 720.9214
as nigger                         

In [32]:
# ============================================================
# CELL 6: EXPERIMENT C — CLASS-AWARE FEATURE SELECTION
# ============================================================

n_classes = len(label_encoder.classes_)

# Number of features we want to finally retain
K_CLASS_AWARE = 5000

# Store Chi-square scores for every class
class_scores = np.zeros(
    (n_classes, X_train_tfidf.shape[1])
)

print("Classes:")
print(label_encoder.classes_)

print("\nCalculating class-specific feature scores...")

Classes:
['age' 'ethnicity' 'gender' 'not_cyberbullying' 'other_cyberbullying'
 'religion']

Calculating class-specific feature scores...


In [33]:
# ============================================================
# ONE-VS-REST CHI-SQUARE SCORING
# ============================================================

for class_id in range(n_classes):

    # Current class = 1
    # All other classes = 0
    y_binary = (y_train == class_id).astype(int)

    scores, p_values = chi2(
        X_train_tfidf,
        y_binary
    )

    class_scores[class_id] = scores

    print(
        f"Class {class_id}: "
        f"{label_encoder.classes_[class_id]}"
    )

print("\nClass-aware scoring completed.")

Class 0: age
Class 1: ethnicity
Class 2: gender
Class 3: not_cyberbullying
Class 4: other_cyberbullying
Class 5: religion

Class-aware scoring completed.


In [34]:
# ============================================================
# CELL 7: CLASS-AWARE FEATURE RANKING
# ============================================================

# For each feature, find its strongest class-specific score
class_aware_scores = np.max(
    class_scores,
    axis=0
)

# Select top K features
class_aware_indices = np.argsort(
    class_aware_scores
)[::-1][:K_CLASS_AWARE]

# Boolean mask
class_aware_mask = np.zeros(
    X_train_tfidf.shape[1],
    dtype=bool
)

class_aware_mask[class_aware_indices] = True

print("Class-aware selected features:",
      class_aware_mask.sum())

Class-aware selected features: 5000


In [35]:
X_train_C = X_train_tfidf[:, class_aware_mask]
X_test_C = X_test_tfidf[:, class_aware_mask]

print("\nExperiment C — Class-aware Selection")
print("--------------------------------------")
print("Train shape:", X_train_C.shape)
print("Test shape :", X_test_C.shape)


Experiment C — Class-aware Selection
--------------------------------------
Train shape: (29851, 5000)
Test shape : (12794, 5000)


In [36]:
X_train_C_combined = hstack([
    X_train_C,
    csr_matrix(X_sent_train.values)
])

X_test_C_combined = hstack([
    X_test_C,
    csr_matrix(X_sent_test.values)
])

print("Final train shape:", X_train_C_combined.shape)
print("Final test shape :", X_test_C_combined.shape)

Final train shape: (29851, 5004)
Final test shape : (12794, 5004)


In [37]:
# ============================================================
# CELL 8: CLASS-SPECIFIC FEATURE ANALYSIS
# ============================================================

for class_id in range(n_classes):

    class_name = label_encoder.classes_[class_id]

    scores = class_scores[class_id]

    top_indices = np.argsort(scores)[::-1][:20]

    print("\n" + "=" * 60)
    print(f"CLASS: {class_name}")
    print("=" * 60)

    for idx in top_indices:
        print(
            f"{feature_names[idx]:40s} "
            f"score = {scores[idx]:.4f}"
        )


CLASS: age
school                                   score = 1930.7148
high                                     score = 1465.3316
high school                              score = 1443.0753
bullied                                  score = 1173.6112
girl                                     score = 1040.8937
bully                                    score = 953.6795
girl bullied                             score = 684.0977
bullied high                             score = 676.5314
school bully                             score = 641.4800
middle school                            score = 286.9810
middle                                   score = 265.8775
girl high                                score = 236.8308
bully school                             score = 221.8210
bully high                               score = 156.2356
got bullied                              score = 152.9843
kid                                      score = 147.5772
bullied girl                             score = 139.70

In [38]:
# ============================================================
# CELL 9: EXPERIMENT D — GLOBAL + CLASS-AWARE
# ============================================================

global_indices = np.where(
    global_selector.get_support()
)[0]

combined_indices = np.union1d(
    global_indices,
    class_aware_indices
)

print("Global features:", len(global_indices))
print("Class-aware features:", len(class_aware_indices))
print("Union features:", len(combined_indices))

Global features: 10000
Class-aware features: 5000
Union features: 10000


In [39]:
X_train_D = X_train_tfidf[:, combined_indices]
X_test_D = X_test_tfidf[:, combined_indices]

print("\nExperiment D — Global + Class-aware")
print("-------------------------------------")
print("Train shape:", X_train_D.shape)
print("Test shape :", X_test_D.shape)


Experiment D — Global + Class-aware
-------------------------------------
Train shape: (29851, 10000)
Test shape : (12794, 10000)


In [40]:
X_train_D_combined = hstack([
    X_train_D,
    csr_matrix(X_sent_train.values)
])

X_test_D_combined = hstack([
    X_test_D,
    csr_matrix(X_sent_test.values)
])

print("Final train shape:", X_train_D_combined.shape)
print("Final test shape :", X_test_D_combined.shape)

Final train shape: (29851, 10004)
Final test shape : (12794, 10004)


In [41]:
# ============================================================
# CELL 10: EXPERIMENT SUMMARY
# ============================================================

print("=" * 70)
print("FEATURE SELECTION EXPERIMENT SUMMARY")
print("=" * 70)

print(f"A. No Selection       : {X_train_A.shape[1]} text features")
print(f"B. Global Selection   : {X_train_B.shape[1]} text features")
print(f"C. Class-aware        : {X_train_C.shape[1]} text features")
print(f"D. Global + Class     : {X_train_D.shape[1]} text features")

FEATURE SELECTION EXPERIMENT SUMMARY
A. No Selection       : 60000 text features
B. Global Selection   : 10000 text features
C. Class-aware        : 5000 text features
D. Global + Class     : 10000 text features


In [45]:
# ============================================================
# NO BALANCING + LINEAR SVM + EVALUATION
# ============================================================

from sklearn.svm import LinearSVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)

# ------------------------------------------------------------
# 1. LINEAR SVM
# ------------------------------------------------------------

svm_none = LinearSVC(
    max_iter=5000,
    random_state=42
)

# ------------------------------------------------------------
# 2. HYPERPARAMETER GRID
# ------------------------------------------------------------

param_grid = {
    'C': [0.01, 0.1, 1, 10]
}

# ------------------------------------------------------------
# 3. GRID SEARCH
# ------------------------------------------------------------

grid_svm_none = GridSearchCV(
    estimator=svm_none,
    param_grid=param_grid,
    cv=5,
    scoring='f1_weighted',
    n_jobs=-1,
    verbose=1
)

# ------------------------------------------------------------
# 4. TRAIN ON ORIGINAL DATA
# ------------------------------------------------------------

print("Training SA + No Balancing + Linear SVM...")

grid_svm_none.fit(
    X_train_A_combined,
    y_train
)

# ------------------------------------------------------------
# 5. BEST MODEL
# ------------------------------------------------------------

best_svm_none = grid_svm_none.best_estimator_

print("\nBest Parameters:")
print(grid_svm_none.best_params_)

print("\nBest Cross-Validation Weighted F1:")
print(f"{grid_svm_none.best_score_:.4f}")

# ------------------------------------------------------------
# 6. PREDICTION
# ------------------------------------------------------------

y_pred_svm_none = best_svm_none.predict(
    X_test_A_combined
)

# ------------------------------------------------------------
# 7. EVALUATION
# ------------------------------------------------------------

svm_none_accuracy = accuracy_score(
    y_test,
    y_pred_svm_none
)

svm_none_weighted_f1 = f1_score(
    y_test,
    y_pred_svm_none,
    average='weighted'
)

svm_none_macro_f1 = f1_score(
    y_test,
    y_pred_svm_none,
    average='macro'
)

# ------------------------------------------------------------
# 8. RESULTS
# ------------------------------------------------------------

print("\n==============================================")
print("NO BALANCING + LINEAR SVM")
print("==============================================")

print(f"Accuracy     : {svm_none_accuracy:.4f}")
print(f"Weighted F1  : {svm_none_weighted_f1:.4f}")
print(f"Macro F1     : {svm_none_macro_f1:.4f}")

# ------------------------------------------------------------
# 9. CLASSIFICATION REPORT
# ------------------------------------------------------------

print("\nClassification Report")
print("-------------------------")

print(
    classification_report(
        y_test,
        y_pred_svm_none,
        target_names=label_encoder.classes_
    )
)

Training SA + No Balancing + Linear SVM...
Fitting 5 folds for each of 4 candidates, totalling 20 fits

Best Parameters:
{'C': 1}

Best Cross-Validation Weighted F1:
0.8435

NO BALANCING + LINEAR SVM
Accuracy     : 0.8499
Weighted F1  : 0.8483
Macro F1     : 0.8284

Classification Report
-------------------------
                     precision    recall  f1-score   support

                age       0.94      0.98      0.96      2387
          ethnicity       0.97      0.99      0.98      2228
             gender       0.90      0.87      0.89      2248
  not_cyberbullying       0.61      0.58      0.60      1934
other_cyberbullying       0.59      0.60      0.60      1608
           religion       0.94      0.96      0.95      2389

           accuracy                           0.85     12794
          macro avg       0.83      0.83      0.83     12794
       weighted avg       0.85      0.85      0.85     12794



In [42]:
# ============================================================
# NO BALANCING + LINEAR SVM + EVALUATION
# ============================================================

from sklearn.svm import LinearSVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)

# ------------------------------------------------------------
# 1. LINEAR SVM
# ------------------------------------------------------------

svm_none = LinearSVC(
    max_iter=5000,
    random_state=42
)

# ------------------------------------------------------------
# 2. HYPERPARAMETER GRID
# ------------------------------------------------------------

param_grid = {
    'C': [0.01, 0.1, 1, 10]
}

# ------------------------------------------------------------
# 3. GRID SEARCH
# ------------------------------------------------------------

grid_svm_none = GridSearchCV(
    estimator=svm_none,
    param_grid=param_grid,
    cv=5,
    scoring='f1_weighted',
    n_jobs=-1,
    verbose=1
)

# ------------------------------------------------------------
# 4. TRAIN ON ORIGINAL DATA
# ------------------------------------------------------------

print("Training SA + No Balancing + Linear SVM...")

grid_svm_none.fit(
    X_train_B_combined,
    y_train
)

# ------------------------------------------------------------
# 5. BEST MODEL
# ------------------------------------------------------------

best_svm_none = grid_svm_none.best_estimator_

print("\nBest Parameters:")
print(grid_svm_none.best_params_)

print("\nBest Cross-Validation Weighted F1:")
print(f"{grid_svm_none.best_score_:.4f}")

# ------------------------------------------------------------
# 6. PREDICTION
# ------------------------------------------------------------

y_pred_svm_none = best_svm_none.predict(
    X_test_B_combined
)

# ------------------------------------------------------------
# 7. EVALUATION
# ------------------------------------------------------------

svm_none_accuracy = accuracy_score(
    y_test,
    y_pred_svm_none
)

svm_none_weighted_f1 = f1_score(
    y_test,
    y_pred_svm_none,
    average='weighted'
)

svm_none_macro_f1 = f1_score(
    y_test,
    y_pred_svm_none,
    average='macro'
)

# ------------------------------------------------------------
# 8. RESULTS
# ------------------------------------------------------------

print("\n==============================================")
print("NO BALANCING + LINEAR SVM")
print("==============================================")

print(f"Accuracy     : {svm_none_accuracy:.4f}")
print(f"Weighted F1  : {svm_none_weighted_f1:.4f}")
print(f"Macro F1     : {svm_none_macro_f1:.4f}")

# ------------------------------------------------------------
# 9. CLASSIFICATION REPORT
# ------------------------------------------------------------

print("\nClassification Report")
print("-------------------------")

print(
    classification_report(
        y_test,
        y_pred_svm_none,
        target_names=label_encoder.classes_
    )
)

Training SA + No Balancing + Linear SVM...
Fitting 5 folds for each of 4 candidates, totalling 20 fits

Best Parameters:
{'C': 1}

Best Cross-Validation Weighted F1:
0.8596

NO BALANCING + LINEAR SVM
Accuracy     : 0.8484
Weighted F1  : 0.8466
Macro F1     : 0.8263

Classification Report
-------------------------
                     precision    recall  f1-score   support

                age       0.94      0.98      0.96      2387
          ethnicity       0.97      0.99      0.98      2228
             gender       0.90      0.87      0.88      2248
  not_cyberbullying       0.60      0.60      0.60      1934
other_cyberbullying       0.60      0.56      0.58      1608
           religion       0.95      0.96      0.96      2389

           accuracy                           0.85     12794
          macro avg       0.83      0.83      0.83     12794
       weighted avg       0.85      0.85      0.85     12794



In [43]:
# ============================================================
# NO BALANCING + LINEAR SVM + EVALUATION
# ============================================================

from sklearn.svm import LinearSVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)

# ------------------------------------------------------------
# 1. LINEAR SVM
# ------------------------------------------------------------

svm_none = LinearSVC(
    max_iter=5000,
    random_state=42
)

# ------------------------------------------------------------
# 2. HYPERPARAMETER GRID
# ------------------------------------------------------------

param_grid = {
    'C': [0.01, 0.1, 1, 10]
}

# ------------------------------------------------------------
# 3. GRID SEARCH
# ------------------------------------------------------------

grid_svm_none = GridSearchCV(
    estimator=svm_none,
    param_grid=param_grid,
    cv=5,
    scoring='f1_weighted',
    n_jobs=-1,
    verbose=1
)

# ------------------------------------------------------------
# 4. TRAIN ON ORIGINAL DATA
# ------------------------------------------------------------

print("Training SA + No Balancing + Linear SVM...")

grid_svm_none.fit(
    X_train_C_combined,
    y_train
)

# ------------------------------------------------------------
# 5. BEST MODEL
# ------------------------------------------------------------

best_svm_none = grid_svm_none.best_estimator_

print("\nBest Parameters:")
print(grid_svm_none.best_params_)

print("\nBest Cross-Validation Weighted F1:")
print(f"{grid_svm_none.best_score_:.4f}")

# ------------------------------------------------------------
# 6. PREDICTION
# ------------------------------------------------------------

y_pred_svm_none = best_svm_none.predict(
    X_test_C_combined
)

# ------------------------------------------------------------
# 7. EVALUATION
# ------------------------------------------------------------

svm_none_accuracy = accuracy_score(
    y_test,
    y_pred_svm_none
)

svm_none_weighted_f1 = f1_score(
    y_test,
    y_pred_svm_none,
    average='weighted'
)

svm_none_macro_f1 = f1_score(
    y_test,
    y_pred_svm_none,
    average='macro'
)

# ------------------------------------------------------------
# 8. RESULTS
# ------------------------------------------------------------

print("\n==============================================")
print("NO BALANCING + LINEAR SVM")
print("==============================================")

print(f"Accuracy     : {svm_none_accuracy:.4f}")
print(f"Weighted F1  : {svm_none_weighted_f1:.4f}")
print(f"Macro F1     : {svm_none_macro_f1:.4f}")

# ------------------------------------------------------------
# 9. CLASSIFICATION REPORT
# ------------------------------------------------------------

print("\nClassification Report")
print("-------------------------")

print(
    classification_report(
        y_test,
        y_pred_svm_none,
        target_names=label_encoder.classes_
    )
)

Training SA + No Balancing + Linear SVM...
Fitting 5 folds for each of 4 candidates, totalling 20 fits

Best Parameters:
{'C': 1}

Best Cross-Validation Weighted F1:
0.8544

NO BALANCING + LINEAR SVM
Accuracy     : 0.8516
Weighted F1  : 0.8502
Macro F1     : 0.8300

Classification Report
-------------------------
                     precision    recall  f1-score   support

                age       0.94      0.98      0.96      2387
          ethnicity       0.97      0.99      0.98      2228
             gender       0.91      0.87      0.89      2248
  not_cyberbullying       0.60      0.63      0.61      1934
other_cyberbullying       0.62      0.55      0.58      1608
           religion       0.95      0.96      0.96      2389

           accuracy                           0.85     12794
          macro avg       0.83      0.83      0.83     12794
       weighted avg       0.85      0.85      0.85     12794



In [44]:
# ============================================================
# NO BALANCING + LINEAR SVM + EVALUATION
# ============================================================

from sklearn.svm import LinearSVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)

# ------------------------------------------------------------
# 1. LINEAR SVM
# ------------------------------------------------------------

svm_none = LinearSVC(
    max_iter=5000,
    random_state=42
)

# ------------------------------------------------------------
# 2. HYPERPARAMETER GRID
# ------------------------------------------------------------

param_grid = {
    'C': [0.01, 0.1, 1, 10]
}

# ------------------------------------------------------------
# 3. GRID SEARCH
# ------------------------------------------------------------

grid_svm_none = GridSearchCV(
    estimator=svm_none,
    param_grid=param_grid,
    cv=5,
    scoring='f1_weighted',
    n_jobs=-1,
    verbose=1
)

# ------------------------------------------------------------
# 4. TRAIN ON ORIGINAL DATA
# ------------------------------------------------------------

print("Training SA + No Balancing + Linear SVM...")

grid_svm_none.fit(
    X_train_D_combined,
    y_train
)

# ------------------------------------------------------------
# 5. BEST MODEL
# ------------------------------------------------------------

best_svm_none = grid_svm_none.best_estimator_

print("\nBest Parameters:")
print(grid_svm_none.best_params_)

print("\nBest Cross-Validation Weighted F1:")
print(f"{grid_svm_none.best_score_:.4f}")

# ------------------------------------------------------------
# 6. PREDICTION
# ------------------------------------------------------------

y_pred_svm_none = best_svm_none.predict(
    X_test_D_combined
)

# ------------------------------------------------------------
# 7. EVALUATION
# ------------------------------------------------------------

svm_none_accuracy = accuracy_score(
    y_test,
    y_pred_svm_none
)

svm_none_weighted_f1 = f1_score(
    y_test,
    y_pred_svm_none,
    average='weighted'
)

svm_none_macro_f1 = f1_score(
    y_test,
    y_pred_svm_none,
    average='macro'
)

# ------------------------------------------------------------
# 8. RESULTS
# ------------------------------------------------------------

print("\n==============================================")
print("NO BALANCING + LINEAR SVM")
print("==============================================")

print(f"Accuracy     : {svm_none_accuracy:.4f}")
print(f"Weighted F1  : {svm_none_weighted_f1:.4f}")
print(f"Macro F1     : {svm_none_macro_f1:.4f}")

# ------------------------------------------------------------
# 9. CLASSIFICATION REPORT
# ------------------------------------------------------------

print("\nClassification Report")
print("-------------------------")

print(
    classification_report(
        y_test,
        y_pred_svm_none,
        target_names=label_encoder.classes_
    )
)

Training SA + No Balancing + Linear SVM...
Fitting 5 folds for each of 4 candidates, totalling 20 fits

Best Parameters:
{'C': 1}

Best Cross-Validation Weighted F1:
0.8596

NO BALANCING + LINEAR SVM
Accuracy     : 0.8484
Weighted F1  : 0.8466
Macro F1     : 0.8263

Classification Report
-------------------------
                     precision    recall  f1-score   support

                age       0.94      0.98      0.96      2387
          ethnicity       0.97      0.99      0.98      2228
             gender       0.90      0.87      0.88      2248
  not_cyberbullying       0.60      0.60      0.60      1934
other_cyberbullying       0.60      0.56      0.58      1608
           religion       0.95      0.96      0.96      2389

           accuracy                           0.85     12794
          macro avg       0.83      0.83      0.83     12794
       weighted avg       0.85      0.85      0.85     12794



## Approach B — Overlap-filtered class-aware selection

In [46]:
# ============================================================
# APPROACH B
# OVERLAP-FILTERED CLASS-AWARE FEATURE SELECTION
# ============================================================

from sklearn.feature_selection import chi2
import numpy as np
import pandas as pd

feature_names = np.array(
    tfidf.get_feature_names_out()
)

n_features = X_train_tfidf.shape[1]
n_classes = len(label_encoder.classes_)

print("Number of features:", n_features)
print("Number of classes :", n_classes)

print("\nClasses:")
for i, name in enumerate(label_encoder.classes_):
    print(i, "→", name)

Number of features: 60000
Number of classes : 6

Classes:
0 → age
1 → ethnicity
2 → gender
3 → not_cyberbullying
4 → other_cyberbullying
5 → religion


In [47]:
# ============================================================
# CLASS-SPECIFIC CHI-SQUARE SCORES
# ============================================================

class_scores = np.zeros(
    (n_classes, n_features)
)

for class_id in range(n_classes):

    y_binary = (
        y_train == class_id
    ).astype(int)

    scores, p_values = chi2(
        X_train_tfidf,
        y_binary
    )

    class_scores[class_id] = scores

print("Class-score matrix shape:")
print(class_scores.shape)

Class-score matrix shape:
(6, 60000)


In [48]:
# ============================================================
# NORMALIZE CLASS-SPECIFIC SCORES
# ============================================================

normalized_scores = np.zeros_like(
    class_scores,
    dtype=float
)

for class_id in range(n_classes):

    scores = class_scores[class_id]

    max_score = np.max(scores)

    if max_score > 0:
        normalized_scores[class_id] = (
            scores / max_score
        )

print(
    "Normalized score range:",
    normalized_scores.min(),
    "to",
    normalized_scores.max()
)

Normalized score range: 6.354629857618167e-15 to 1.0


In [49]:
# ============================================================
# INITIAL CLASS-AWARE CANDIDATES
# ============================================================

K_PER_CLASS = 3000

candidate_indices = set()

for class_id in range(n_classes):

    top_indices = np.argsort(
        normalized_scores[class_id]
    )[::-1][:K_PER_CLASS]

    candidate_indices.update(
        top_indices
    )

candidate_indices = np.array(
    sorted(candidate_indices)
)

print(
    "Initial class-aware candidate features:",
    len(candidate_indices)
)

Initial class-aware candidate features: 16546


In [50]:
# ============================================================
# IDENTIFY IMPORTANT CLASSES
# ============================================================

classes = list(label_encoder.classes_)

print(classes)

['age', 'ethnicity', 'gender', 'not_cyberbullying', 'other_cyberbullying', 'religion']


In [51]:
not_cb_id = classes.index(
    'not_cyberbullying'
)

other_cb_id = classes.index(
    'other_cyberbullying'
)

print("Not-CB ID  :", not_cb_id)
print("Other-CB ID:", other_cb_id)

Not-CB ID  : 3
Other-CB ID: 4


In [52]:
# ============================================================
# OVERLAP + DISCRIMINATION
# ============================================================

eps = 1e-10

s1 = normalized_scores[
    not_cb_id,
    candidate_indices
]

s2 = normalized_scores[
    other_cb_id,
    candidate_indices
]

# Degree of overlap
overlap_score = (
    np.minimum(s1, s2)
    /
    (np.maximum(s1, s2) + eps)
)

# Class discrimination
discrimination_score = np.abs(
    s1 - s2
)

print("Overlap range:",
      overlap_score.min(),
      "to",
      overlap_score.max())

print("Discrimination range:",
      discrimination_score.min(),
      "to",
      discrimination_score.max())

Overlap range: 2.534461187753878e-10 to 0.9969632701599177
Discrimination range: 8.004020766028118e-08 to 0.99403094249306


In [53]:
# ============================================================
# INSPECT OVERLAP/DISCRIMINATION
# ============================================================

B_analysis = pd.DataFrame({
    'feature': feature_names[candidate_indices],
    'not_cb_score': s1,
    'other_cb_score': s2,
    'overlap': overlap_score,
    'discrimination': discrimination_score
})

B_analysis = B_analysis.sort_values(
    'overlap',
    ascending=False
)

B_analysis.head(30)

,feature,not_cb_score,other_cb_score,overlap,discrimination
7350,hoe,0.003221,0.003212,0.996963,0.000010
12935,schedule,0.004005,0.003992,0.996893,0.000012
1278,bloke,0.004103,0.004089,0.996801,0.000013
9769,miserable,0.001142,0.001149,0.994403,0.000006
15540,use,0.004045,0.004070,0.993827,0.000025
438,antigay,0.003436,0.003457,0.993710,0.000022
16262,would call,0.002118,0.002094,0.988696,0.000024
5151,even,0.005868,0.005787,0.986240,0.000081
1336,bottle,0.004674,0.004593,0.982589,0.000081
4809,dont start,0.000243,0.000239,0.982529,0.000004


In [54]:
OVERLAP_THRESHOLD = 0.90
DISCRIMINATION_THRESHOLD = 0.10

In [55]:
# ============================================================
# FILTER LOW-DISCRIMINATION OVERLAP
# ============================================================

remove_mask = (
    (overlap_score > OVERLAP_THRESHOLD)
    &
    (discrimination_score < DISCRIMINATION_THRESHOLD)
)

removed_features = candidate_indices[
    remove_mask
]

kept_candidates = candidate_indices[
    ~remove_mask
]

print("Initial candidates :", len(candidate_indices))
print("Removed features   :", len(removed_features))
print("Remaining features :", len(kept_candidates))

Initial candidates : 16546
Removed features   : 102
Remaining features : 16444


In [56]:
# ============================================================
# FINAL B RANKING
# ============================================================

# Maximum class-specific importance
class_aware_strength = np.max(
    normalized_scores[:, kept_candidates],
    axis=0
)

ranking = np.argsort(
    class_aware_strength
)[::-1]

K_FINAL = 5000

final_B_indices = kept_candidates[
    ranking[:K_FINAL]
]

print(
    "Final Approach B features:",
    len(final_B_indices)
)

Final Approach B features: 5000


In [57]:
# ============================================================
# APPROACH B — FINAL FEATURE MATRICES
# ============================================================

X_train_B = X_train_tfidf[
    :,
    final_B_indices
]

X_test_B = X_test_tfidf[
    :,
    final_B_indices
]

print("B text train:", X_train_B.shape)
print("B text test :", X_test_B.shape)

B text train: (29851, 5000)
B text test : (12794, 5000)


In [58]:
from scipy.sparse import csr_matrix, hstack

X_train_B_combined = hstack([
    X_train_B,
    csr_matrix(X_sent_train.values)
])

X_test_B_combined = hstack([
    X_test_B,
    csr_matrix(X_sent_test.values)
])

print("B final train:", X_train_B_combined.shape)
print("B final test :", X_test_B_combined.shape)

B final train: (29851, 5004)
B final test : (12794, 5004)


In [59]:
# ============================================================
# NO BALANCING + LINEAR SVM + EVALUATION
# ============================================================

from sklearn.svm import LinearSVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)

# ------------------------------------------------------------
# 1. LINEAR SVM
# ------------------------------------------------------------

svm_none = LinearSVC(
    max_iter=5000,
    random_state=42
)

# ------------------------------------------------------------
# 2. HYPERPARAMETER GRID
# ------------------------------------------------------------

param_grid = {
    'C': [0.01, 0.1, 1, 10]
}

# ------------------------------------------------------------
# 3. GRID SEARCH
# ------------------------------------------------------------

grid_svm_none = GridSearchCV(
    estimator=svm_none,
    param_grid=param_grid,
    cv=5,
    scoring='f1_weighted',
    n_jobs=-1,
    verbose=1
)

# ------------------------------------------------------------
# 4. TRAIN ON ORIGINAL DATA
# ------------------------------------------------------------

print("Training SA + No Balancing + Linear SVM...")

grid_svm_none.fit(
    X_train_B_combined,
    y_train
)

# ------------------------------------------------------------
# 5. BEST MODEL
# ------------------------------------------------------------

best_svm_none = grid_svm_none.best_estimator_

print("\nBest Parameters:")
print(grid_svm_none.best_params_)

print("\nBest Cross-Validation Weighted F1:")
print(f"{grid_svm_none.best_score_:.4f}")

# ------------------------------------------------------------
# 6. PREDICTION
# ------------------------------------------------------------

y_pred_svm_none = best_svm_none.predict(
    X_test_B_combined
)

# ------------------------------------------------------------
# 7. EVALUATION
# ------------------------------------------------------------

svm_none_accuracy = accuracy_score(
    y_test,
    y_pred_svm_none
)

svm_none_weighted_f1 = f1_score(
    y_test,
    y_pred_svm_none,
    average='weighted'
)

svm_none_macro_f1 = f1_score(
    y_test,
    y_pred_svm_none,
    average='macro'
)

# ------------------------------------------------------------
# 8. RESULTS
# ------------------------------------------------------------

print("\n==============================================")
print("NO BALANCING + LINEAR SVM")
print("==============================================")

print(f"Accuracy     : {svm_none_accuracy:.4f}")
print(f"Weighted F1  : {svm_none_weighted_f1:.4f}")
print(f"Macro F1     : {svm_none_macro_f1:.4f}")

# ------------------------------------------------------------
# 9. CLASSIFICATION REPORT
# ------------------------------------------------------------

print("\nClassification Report")
print("-------------------------")

print(
    classification_report(
        y_test,
        y_pred_svm_none,
        target_names=label_encoder.classes_
    )
)

Training SA + No Balancing + Linear SVM...
Fitting 5 folds for each of 4 candidates, totalling 20 fits

Best Parameters:
{'C': 1}

Best Cross-Validation Weighted F1:
0.8573

NO BALANCING + LINEAR SVM
Accuracy     : 0.8453
Weighted F1  : 0.8430
Macro F1     : 0.8220

Classification Report
-------------------------
                     precision    recall  f1-score   support

                age       0.93      0.98      0.95      2387
          ethnicity       0.96      0.99      0.98      2228
             gender       0.91      0.85      0.88      2248
  not_cyberbullying       0.58      0.65      0.61      1934
other_cyberbullying       0.63      0.50      0.56      1608
           religion       0.95      0.96      0.96      2389

           accuracy                           0.85     12794
          macro avg       0.83      0.82      0.82     12794
       weighted avg       0.84      0.85      0.84     12794



In [ ]:
"The study investigates feature representations and discriminative feature selection for fine-grained cyberbullying detection, with particular emphasis on hard-negative and heterogeneous classes."